<a href="https://colab.research.google.com/github/jgromero/drl-csic/blob/main/code/from-scratch/cartpole-reinforce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# REINFORCE for _cartpole_

[**Juan Gómez Romero**](https://ccia.ugr.es/~jgomez)  
Departamento de Ciencias de la Computación e Inteligencia Artificial  
Universidad de Granada  
This work is licensed under the [Apache License 2.0](https://choosealicense.com/licenses/apache-2.0/).

---
Based on:
> Udacity, Deep Reinforcement Learning Course. Available on [GitHub](https://github.com/udacity/deep-reinforcement-learning/tree/master/reinforce).

## Configuration

In [ ]:
%pip install "gymnasium[classic-control]"

In [2]:
# set render to 'rgb_array' (change to 'human' for local display-enabled runtime)
render_mode = "rgb_array"

## Environment

We will be using [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/), a simple control problem.

<img src="https://github.com/jgromero/drl-csic/blob/main/code/cartpole.gif?raw=true"/>


The action and observation spaces can be printed as follows.

In [3]:
import gymnasium as gym
env = gym.make("CartPole-v1", render_mode = render_mode)
print(env.observation_space)
print(env.action_space)

Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Discrete(2)


## REINFORCE algorithm
Next we provide a generic implementation of REINFORCE. We follow the same structure as in [cartpole-dqn.ipynb](https://github.com/jgromero/drl-csic/blob/main/code/from-scratch/cartpole-dqn.ipynb), but, since the code is much shorter, we include it in the notebook instead of in external files:

*   `Policy`: Implementation of the neural network of the agent (the policy network), equivalent to [model.py](https://github.com/jgromero/drl-csic/blob/main/code/from-scratch/model.py) in DQN.
*   `Agent`: Implementation of the agent capabilities (act, step, learn), equivalent to [dqn_agent.py](https://github.com/jgromero/drl-csic/blob/main/code/from-scratch/dqn_agent.py) in DQN.

The [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/) environment is considered solved when the agent obtains an average score of at least $475$ points over 100 consecutive episodes. Episodes are truncated at $500$ steps, which is therefore the maximum score.

### Imports and initializations

In [ ]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
import numpy as np
from collections import deque
import matplotlib.pyplot as plt
%matplotlib inline

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

### Policy network
Unlike the Q-network of DQN, which estimates the value of each action, the policy network directly outputs the probability of selecting each action.

In [ ]:
class Policy(nn.Module):
    """Policy network: maps states to action probabilities."""

    def __init__(self, state_size, action_size, seed, fc1_units=16):
        """Initialize parameters and build model.
        Params
        ======
            state_size (int): Dimension of each state
            action_size (int): Dimension of each action
            seed (int): Random seed
            fc1_units (int): Number of nodes in the hidden layer
        """
        super(Policy, self).__init__()
        torch.manual_seed(seed)
        self.fc1 = nn.Linear(state_size, fc1_units)
        self.fc2 = nn.Linear(fc1_units, action_size)

    def forward(self, state):
        """Build a network that maps state -> action probabilities."""
        x = F.relu(self.fc1(state))
        return F.softmax(self.fc2(x), dim=1)

### Agent
The main difference with the DQN agent is _when_ it learns. The DQN agent learns at every step from a mini-batch of past experiences stored in the replay buffer. The REINFORCE agent only stores the log-probability of each action and the reward obtained (`step`), and learns at the end of each episode (`learn`), when the discounted return from each step onwards, $G_t$, can be calculated.

To reduce the variance of the gradient estimate, each action is weighted by the (normalized) $G_t$ instead of by the total return of the episode.

In [ ]:
GAMMA = 0.99            # discount factor (gamma)
LR = 1e-2               # learning rate (eta)

class Agent():
    """Interacts with and learns from the environment."""

    def __init__(self, state_size, action_size, seed):
        """Initialize an Agent object.

        Params
        ======
            state_size (int): dimension of each state
            action_size (int): dimension of each action
            seed (int): random seed
        """
        self.state_size = state_size
        self.action_size = action_size
        random.seed(seed)

        # Policy network
        self.policy = Policy(state_size, action_size, seed).to(device)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=LR)

        # Trajectory of the current episode
        self.saved_log_probs = []  # log-probabilities of the actions taken
        self.rewards = []          # rewards obtained

    def act(self, state):
        """Samples an action for given state as per current policy.

        Params
        ======
            state (array_like): current state
        Returns
        =======
            action (int): action sampled
            log_prob (torch.Tensor): log-probability of the action (needed to learn)
        """
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.policy(state).cpu()
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action)

    def step(self, log_prob, reward):
        """Stores the log-probability of the action taken (At) and the reward obtained (Rt+1).

        Params
        ======
            log_prob (torch.Tensor): log-probability of the action taken
            reward (float): reward obtained
        """
        self.saved_log_probs.append(log_prob)
        self.rewards.append(reward)

    def learn(self, gamma=GAMMA):
        """Update policy parameters using the trajectory of the episode that has just finished.

        Params
        ======
            gamma (float): discount factor
        """
        # calculate discounted return from each step onwards: G_t = R_t+1 + gamma * G_t+1
        returns = deque()
        G = 0
        for reward in reversed(self.rewards):
            G = reward + gamma * G
            returns.appendleft(G)

        # normalize returns (reduces the variance of the gradient estimate)
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + 1e-9)

        # compute loss & update
        policy_loss = []
        for log_prob, G in zip(self.saved_log_probs, returns):
            policy_loss.append(-log_prob * G)
        policy_loss = torch.cat(policy_loss).sum()

        self.optimizer.zero_grad()
        policy_loss.backward()
        self.optimizer.step()

        # start a new trajectory
        self.saved_log_probs = []
        self.rewards = []

### Training loop
The training loop is almost the same as in DQN. The only difference is the call to `agent.learn()` at the end of each episode.

In [ ]:
SOLVED_SCORE = 475.0   # average score (last 100 episodes) to consider CartPole-v1 solved

def reinforce(agent, n_episodes=1000, max_t=500, print_every=100):
    """REINFORCE.

    Params
    ======
        agent (Agent): agent to train
        n_episodes (int): maximum number of episodes (n_episodes)
        max_t (int): maximum number of steps per episode (n_steps)
        print_every (int): print score each print_every episodes
    """
    scores = []                          # scores for every episode
    scores_window = deque(maxlen=100)    # scores for the last 100 episodes

    for i_episode in range(1, n_episodes+1):
        state, _ = env.reset()
        score = 0

        for t in range(max_t):

            # choose action At with the policy network
            action, log_prob = agent.act(state)

            # apply At and get Rt+1, St+1
            next_state, reward, terminated, truncated, _ = env.step(action)

            # store log-probability of At and Rt+1
            agent.step(log_prob, reward)

            # advance to next state
            state = next_state
            score += reward

            if terminated or truncated:
                break

        # train & update with the whole trajectory
        agent.learn()

        scores_window.append(score)       # save last score (last 100)
        scores.append(score)              # save last score (every episode)

        # print training info
        if i_episode % print_every == 0:
            print('Episode {}\tAverage score (last 100): {:.2f}'.format(i_episode, np.mean(scores_window)))
        if len(scores_window) == 100 and np.mean(scores_window) >= SOLVED_SCORE:
            print('\nProblem solved in {:d} episodes!\tAverage score (last 100): {:.2f}'.format(i_episode-100, np.mean(scores_window)))
            break
    else:
        print('\nProblem not solved in {:d} episodes.\tAverage score (last 100): {:.2f}'.format(n_episodes, np.mean(scores_window)))

    torch.save(agent.policy.state_dict(), 'checkpoint-cartpole-reinforce.pth') # save weights for trained agent

    return scores

agent = Agent(state_size=4, action_size=2, seed=0)
scores = reinforce(agent, n_episodes=5000)

# plot the scores
fig = plt.figure()
ax = fig.add_subplot(111)
plt.plot(np.arange(len(scores)), scores)
plt.ylabel('Score')
plt.xlabel('Episode #')
plt.show()

## Visualize trained agent

We can afterwards load the weights into the agent and visualize how it acts on the environment. (Local run is required for 2D rendering, not Google Colaboratory.)

In [ ]:
# load weights from `checkpoint-cartpole-reinforce.pth`
agent_load = Agent(state_size=4, action_size=2, seed=0)

agent_load.policy.load_state_dict(torch.load('checkpoint-cartpole-reinforce.pth', map_location=device, weights_only=True))

for i in range(3):
    state, _ = env.reset()
    for j in range(500):
        action, _ = agent_load.act(state)
        env.render()
        state, reward, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            break

env.close()